In [1]:
import requests, pandas as pd, json
from IPython.display import display

API = "http://localhost:8001/api/v1"
TOP_K = 3  # retrieve top-K to compute recall@5, @10, @20

In [2]:
# GET /api/v1/associations → list of {species_a, species_b, danger_note, source}
resp = requests.get(f"{API}/associations")
resp.raise_for_status()
pairs = resp.json()
print(f"{len(pairs)} ground truth pairs loaded")
pd.DataFrame(pairs)

31 ground truth pairs loaded


,species_a,species_b,danger_note,source
0,Amanita caesarea,Amanita muscaria,A. caesarea is choice edible; A. muscaria is t...,European field guides
1,Amanita caesarea,Amanita phalloides,A. caesarea is choice edible; A. phalloides is...,European field guides
2,Agaricus campestris,Agaricus xanthodermus,A. campestris is edible; A. xanthodermus cause...,European field guides
3,Agaricus campestris,Amanita phalloides,A. campestris is edible; A. phalloides is dead...,Multiple sources
4,Cantharellus cibarius,Omphalotus olearius,C. cibarius is choice edible; O. olearius is t...,Global field guides
5,Cantharellus cibarius,Hygrophoropsis aurantiaca,C. cibarius is edible; H. aurantiaca is inedib...,European field guides
6,Boletus edulis,Tylopilus felleus,B. edulis is choice edible; T. felleus is extr...,European field guides
7,Macrolepiota procera,Chlorophyllum molybdites,M. procera is edible; C. molybdites is toxic. ...,Global field guides
8,Morchella esculenta,Gyromitra esculenta,M. esculenta is choice edible; G. esculenta is...,Global field guides
9,Kuehneromyces mutabilis,Galerina marginata,K. mutabilis is edible; G. marginata is deadly...,European field guides


In [3]:
def query_lookalikes(name, top_k=TOP_K, w_morph=0.60, w_eco=0.25, w_taxon=0.15):
    """POST /api/v1/lookalikes — returns list of candidate dicts or raises."""
    r = requests.post(f"{API}/lookalikes", json={
        "species_name": name,
        "top_k": top_k,
        "weight_morphological": w_morph,
        "weight_ecological": w_eco,
        "weight_taxonomic": w_taxon,
    })
    if r.status_code != 200:
        return None, r.text
    data = r.json()
    return data["candidates"], None

In [4]:
# For each pair (A,B):
#   - query lookalikes of A, check if B is in top-K → record rank, scores
#   - query lookalikes of B, check if A is in top-K → record rank, scores
# Produces rows: pair, direction, rank_found (None=miss), sim_overall, sim_morph, sim_eco, sim_taxon

def evaluate_pairs(pairs, top_k=TOP_K, **weight_kwargs):
    rows = []
    for p in pairs:
        for query, target in [(p["species_a"], p["species_b"]),
                               (p["species_b"], p["species_a"])]:
            candidates, err = query_lookalikes(query, top_k=top_k, **weight_kwargs)
            if err:
                rows.append({"query": query, "target": target,
                             "danger_note": p.get("danger_note"), "rank": None,
                             "hit": False, "error": err})
                continue
            names = [c["scientific_name"] for c in candidates]
            rank = names.index(target) + 1 if target in names else None
            cand = next((c for c in candidates if c["scientific_name"] == target), {})
            rows.append({
                "query": query, "target": target,
                "danger_note": p.get("danger_note"),
                "rank": rank,
                "hit": rank is not None,
                "sim_overall": cand.get("similarity_overall"),
                "sim_morph":   cand.get("similarity_morphological"),
                "sim_eco":     cand.get("similarity_ecological"),
                "sim_taxon":   cand.get("similarity_taxonomic"),
            })
    return pd.DataFrame(rows)

df = evaluate_pairs(pairs)
df

,query,target,danger_note,rank,hit,sim_overall,sim_morph,sim_eco,sim_taxon,error
0,Amanita caesarea,Amanita muscaria,A. caesarea is choice edible; A. muscaria is t...,1.0,True,0.8949,0.9150,0.9289,0.7577,NaN
1,Amanita muscaria,Amanita caesarea,A. caesarea is choice edible; A. muscaria is t...,2.0,True,0.8949,0.9150,0.9289,0.7577,NaN
2,Amanita caesarea,Amanita phalloides,A. caesarea is choice edible; A. phalloides is...,NaN,False,NaN,NaN,NaN,NaN,NaN
3,Amanita phalloides,Amanita caesarea,A. caesarea is choice edible; A. phalloides is...,3.0,True,0.8529,0.8527,0.9466,0.6976,NaN
4,Agaricus campestris,Agaricus xanthodermus,A. campestris is edible; A. xanthodermus cause...,1.0,True,0.8822,0.8451,0.9688,0.8863,NaN
...,...,...,...,...,...,...,...,...,...,...
57,Cortinarius Seriocybe,Laccaria amethystina,None,NaN,False,NaN,NaN,NaN,NaN,"{""detail"":""Species not found: 'Cortinarius Ser..."
58,Cortinarius Violaceus,Cortinarius Seriocybe,None,NaN,False,NaN,NaN,NaN,NaN,NaN
59,Cortinarius Seriocybe,Cortinarius Violaceus,None,NaN,False,NaN,NaN,NaN,NaN,"{""detail"":""Species not found: 'Cortinarius Ser..."
60,Russula emetica,Russula Vesca,None,NaN,False,NaN,NaN,NaN,NaN,NaN


In [7]:
# Recall@K
for k in [1, 2, 3, 5]:
    hit_at_k = (df["rank"] <= k).sum()
    print(f"Recall@{k:2d}: {hit_at_k}/{len(df)} = {hit_at_k/len(df):.1%}")

# Misses
print("\nMisses (target not found in top-20):")
display(df[~df["hit"]][["query", "target", "danger_note"]])

Recall@ 1: 8/62 = 12.9%
Recall@ 2: 11/62 = 17.7%
Recall@ 3: 14/62 = 22.6%
Recall@ 5: 14/62 = 22.6%

Misses (target not found in top-20):


,query,target,danger_note
2,Amanita caesarea,Amanita phalloides,A. caesarea is choice edible; A. phalloides is...
6,Agaricus campestris,Amanita phalloides,A. campestris is edible; A. phalloides is dead...
7,Amanita phalloides,Agaricus campestris,A. campestris is edible; A. phalloides is dead...
8,Cantharellus cibarius,Omphalotus olearius,C. cibarius is choice edible; O. olearius is t...
9,Omphalotus olearius,Cantharellus cibarius,C. cibarius is choice edible; O. olearius is t...
10,Cantharellus cibarius,Hygrophoropsis aurantiaca,C. cibarius is edible; H. aurantiaca is inedib...
11,Hygrophoropsis aurantiaca,Cantharellus cibarius,C. cibarius is edible; H. aurantiaca is inedib...
14,Macrolepiota procera,Chlorophyllum molybdites,M. procera is edible; C. molybdites is toxic. ...
15,Chlorophyllum molybdites,Macrolepiota procera,M. procera is edible; C. molybdites is toxic. ...
16,Morchella esculenta,Gyromitra esculenta,M. esculenta is choice edible; G. esculenta is...


In [6]:
display(
    df[df["hit"]].sort_values("rank")
      [["query", "target", "rank", "sim_overall", "sim_morph", "sim_eco", "sim_taxon"]]
      .round(3)
)

,query,target,rank,sim_overall,sim_morph,sim_eco,sim_taxon
0,Amanita caesarea,Amanita muscaria,1.0,0.895,0.915,0.929,0.758
4,Agaricus campestris,Agaricus xanthodermus,1.0,0.882,0.845,0.969,0.886
12,Boletus edulis,Tylopilus felleus,1.0,0.864,0.845,0.969,0.769
13,Tylopilus felleus,Boletus edulis,1.0,0.864,0.845,0.969,0.769
23,Amanita pantherina,Amanita rubescens,1.0,0.878,0.906,0.876,0.767
24,Lactarius deliciosus,Lactarius torminosus,1.0,0.859,0.840,0.917,0.836
25,Lactarius torminosus,Lactarius deliciosus,1.0,0.859,0.840,0.917,0.836
26,Coprinus comatus,Coprinopsis atramentaria,1.0,0.849,0.814,0.944,0.829
1,Amanita muscaria,Amanita caesarea,2.0,0.895,0.915,0.929,0.758
5,Agaricus xanthodermus,Agaricus campestris,2.0,0.882,0.845,0.969,0.886


In [8]:
# Compare recall@10 across different weight configs
configs = {
    "default  (60/25/15)": dict(w_morph=0.60, w_eco=0.25, w_taxon=0.15),
    "morph-heavy (80/10/10)": dict(w_morph=0.80, w_eco=0.10, w_taxon=0.10),
    "balanced (50/30/20)": dict(w_morph=0.50, w_eco=0.30, w_taxon=0.20),
    "taxon-heavy (40/20/40)": dict(w_morph=0.40, w_eco=0.20, w_taxon=0.40),
}

summary = {}
for label, w in configs.items():
    df_w = evaluate_pairs(pairs, top_k=10, **w)
    summary[label] = {f"R@{k}": (df_w["rank"] <= k).mean() for k in [5, 10]}

pd.DataFrame(summary).T.round(3)

KeyboardInterrupt: 

In [ ]:
# For misses: show where the target actually ranks across group embeddings individually
# (re-query with each group at 100% weight to diagnose which dimension is failing)
misses = df[~df["hit"]][["query","target","danger_note"]].to_dict("records")
for m in misses:
    print(f"\n{m['query']} → {m['target']}  ({m['danger_note']})")
    for label, w in [("morph-only", (1,0,0)), ("eco-only", (0,1,0)), ("taxon-only", (0,0,1))]:
        cands, _ = query_lookalikes(m["query"], top_k=58, w_morph=w[0], w_eco=w[1], w_taxon=w[2])
        if cands:
            names = [c["scientific_name"] for c in cands]
            rank = names.index(m["target"]) + 1 if m["target"] in names else ">58"
            print(f"  {label}: rank {rank}")